## Run Evaluation

In [ ]:
import json
import pandas as pd
from evaluation import compute_metrics
from eval_energy import evaluate

EXPERIMENTS = {
    "SBERT": {
        "Efthymiou":           "DART/exp/sbert/eft_sbert_multilingual.pkl",
        "T2Dv2":               "DART/exp/sbert/t2dv2_sbert_multilingual.pkl",
        "Energy domain":       "DART/exp/sbert/energy_sbert_multilingual.pkl",
    },
    "multilingual-e5-base": {
        "Efthymiou":           "DART/exp/e5/eft_e5_multilingual.pkl",
        "T2Dv2":               "DART/exp/e5/t2dv2_e5_multilingual.pkl",
        "Energy domain":       "DART/exp/e5/energy_e5_multilingual.pkl",
    },
    "DART (Bi-Encoder)": {
        "Efthymiou":           "DART/exp/dart_encoder/eft_retrieve.pkl",
        "T2Dv2":               "DART/exp/dart_encoder/T2Dv2_retrieval_result.pkl",
        "Energy domain":       "DART/exp/dart_encoder/energy_retrieve.pkl",
    },
    "Gemma-4-31B": {
        "Efthymiou":           "DART/exp/gemma/eft_gemma_final_results.pkl",
        "T2Dv2":               "DART/exp/gemma/t2dv2_gemma_final_results.pkl",
        "Energy domain":       "DART/exp/gemma/energy_gemma_results.pkl",
    },
    "GPT-4o-mini": {
        "Efthymiou":           "DART/exp/gpt/gpt/eft_gpt_final_results.pkl",
        "T2Dv2":               "DART/exp/gpt/gpt/t2dv2_gpt_final_results.pkl",
        "Energy domain":       "DART/exp/gpt/gpt/energy_gpt_final_results.pkl",
    },
    "GPT-4o-mini (CoT)": {
        "Efthymiou":           "DART/exp/gpt/cot/eft_cot_final_results.pkl",
        "T2Dv2":               "DART/exp/gpt/cot/t2dv2_cot_final_results.pkl",
        "Energy domain":       "DART/exp/gpt/cot/energy_cot_final_results.pkl",
    },
    "DART": {
        "Efthymiou":           "DART/exp/dart_full/eft_dart_final_results.pkl",
        "T2Dv2":               "DART/exp/dart_full/t2dv2_dart_final_results.pkl",
        "Energy domain":       "DART/exp/dart_full/energy_dart_final_results.pkl",
    },
}

DATASETS = ["Efthymiou", "T2Dv2"]
METRICS  = ["MRR", "Hit@1", "Hit@10"]

ONTOLOGIES = {
    "Efthymiou":     "DART/data/ontology/dbo_cache.json",
    "T2Dv2":         "DART/data/ontology/dbo_cache.json",
    "Energy domain": "DART/data/ontology/bed_cache.json",
}

DATASETS = ["Efthymiou", "T2Dv2", "Energy domain"]
METRICS  = ["MRR", "Hit@1", "Hit@10"]

_ontology_cache = {}

def load_ontology(path: str) -> dict:
    if path not in _ontology_cache:
        with open(path) as f:
            raw = json.load(f)
        _ontology_cache[path] = raw.get("concepts", raw)
    return _ontology_cache[path]


# all_results[model][dataset] = {MRR, Hit@1, Hit@10}
all_results = {model: {} for model in EXPERIMENTS}

# for model, datasets in EXPERIMENTS.items():
#     for dataset, pkl_path in datasets.items():
#         ontology = load_ontology(ONTOLOGIES[dataset])
#         print(f"  {model} / {dataset} ...")
#         pkl_path = pkl_path[9:]  # Remove 'DART/exp/' prefix to make path relative to current directory
#         output = compute_metrics(
#             results_pkl=pkl_path,
#             ontology=ontology,
#             top_n=10,
#             output_json=None,
#         )
#         exact = output["approximate_hierarchical"]
#         all_results[model][dataset] = {
#             "MRR":   exact["AH-MRR"],
#             "Hit@1": exact["AH-Hit@1"],
#             "Hit@10":exact["AH-Hit@10"],
#         }

# print("Done.")


GT_CSV_MAP = {
    "Energy domain": "/home/iai/dg3485/CTA/iswc/2026/data/energy/ground_truth_filter_copy.csv",
}

DBO_DATASETS = {"Efthymiou", "T2Dv2"}
BED_DATASETS = {"Energy domain"}
for model, datasets in EXPERIMENTS.items():
    for dataset, pkl_path in datasets.items():
        pkl_path = pkl_path[9:]  # Remove 'DART/exp/' prefix

        if dataset in DBO_DATASETS:
            ontology = load_ontology("../data/ontology/dbo_cache.json")
            print(f"  {model} / {dataset} ...")
            output = compute_metrics(
                results_pkl=pkl_path,
                ontology=ontology,
                top_n=10,
                output_json=None,
            )
            ah = output["approximate_hierarchical"]
            all_results[model][dataset] = {
                "MRR":    ah["AH-MRR"],
                "Hit@1":  ah["AH-Hit@1"],
                "Hit@10": ah["AH-Hit@10"],
            }
        else:
            ontology = load_ontology("../data/ontology/bed_cache.json")
            gt_csv   = GT_CSV_MAP[dataset]
            print(f"  {model} / {dataset} ...")
            import argparse
            args = argparse.Namespace(
                results_pkl=pkl_path,
                gt_csv=GT_CSV_MAP[dataset],
                ontology="../data/ontology/bed_cache.json",
                output_json=None,
                top_n=10,
            )
            # output = evaluate(
            #     results_pkl=pkl_path,
            #     gt_csv=gt_csv,
            #     ontology=ontology,
            #     top_n=10,
            #     output_json=None,
            # )
            output = evaluate(args)

            all_results[model][dataset] = {
                "MRR":    round(output.get("AH-MRR",   0.0), 4),
                "Hit@1":  round(output.get("AH-Hit@1", 0.0), 4),
                "Hit@10": round(output.get("AH-Hit@10",0.0), 4),
            }

print("Done.")


## Overall Metrics

In [63]:
# Split datasets by ontology
DBO_DATASETS = [d for d in DATASETS if d in {"Efthymiou", "T2Dv2"}]
BED_DATASETS = [d for d in DATASETS if d in {"Energy domain"}]

# DBO metrics: MRR, Hit@1, Hit@10
DBO_METRICS = ["MRR", "Hit@1", "Hit@10"]

# BED metrics: same ranking metrics
BED_METRICS = ["MRR", "Hit@1", "Hit@10"]


def build_table(datasets, metrics):
    col_tuples = [(d, m) for d in datasets for m in metrics]
    rows = []
    for model in EXPERIMENTS:
        row = {}
        for dataset, metric in col_tuples:
            row[(dataset, metric)] = all_results[model].get(dataset, {}).get(metric, float("nan"))
        rows.append(row)
    df = pd.DataFrame(rows, index=list(EXPERIMENTS.keys()))
    df.columns = pd.MultiIndex.from_tuples(col_tuples)
    df.index.name = "Models"
    return df


df_dbo = build_table(DBO_DATASETS, DBO_METRICS)
df_bed = build_table(BED_DATASETS, BED_METRICS)

print("Efthymiou, T2Dv2")
print(df_dbo.to_string())
print()
print("Energy Domain Dataset=")
print(df_bed.to_string())


Efthymiou, T2Dv2
                     Efthymiou                   T2Dv2                
                           MRR   Hit@1  Hit@10     MRR   Hit@1  Hit@10
Models                                                                
SBERT                   0.4248  0.3285  0.6225  0.5633  0.4861  0.7269
multilingual-e5-base    0.3411  0.2541  0.5281  0.3147  0.2037  0.5694
DART (Bi-Encoder)       0.5837  0.4764  0.7793  0.7391  0.6759  0.8565
Gemma-4-31B             0.7225  0.6588  0.8203  0.8266  0.7824  0.9028
GPT-4o-mini             0.6972  0.6225  0.8167  0.8337  0.7917  0.9028
GPT-4o-mini (CoT)       0.6939  0.6171  0.8131  0.8240  0.7870  0.8843
DART                    0.7276  0.6661  0.8203  0.8924  0.8611  0.9398

Energy Domain Dataset=
                     Energy domain                
                               MRR   Hit@1  Hit@10
Models                                            
SBERT                       0.3749  0.2056  0.6916
multilingual-e5-base        0.2974  0.2022  0